In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [2]:
transform=transforms.Compose([
    transforms.Resize((128,128)),  # Resize all Images In fixed Sizes
    transforms.ToTensor()        # Convert Image to numbers(0-1) normalization   
])

In [3]:
train_data=datasets.ImageFolder(
    root="train",
    transform=transform
)

In [4]:
val_data = datasets.ImageFolder(
    root='val',
    transform=transform
)

In [5]:
print(len(train_data))
print(len(val_data))

275
70


In [6]:
print(train_data.classes)

['cat', 'dog']


In [7]:
image, label = train_data[0]

print(image.shape)
print(label)

torch.Size([3, 128, 128])
0


In [8]:
train_loader=DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

In [9]:
val_loader=DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

In [10]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [24]:
from torchvision.models import resnet50,ResNet50_Weights

# Load pre-trained ResNet50
model = resnet50(weights=ResNet50_Weights.DEFAULT)

# Freeze early layers (keep ImageNet knowledge)
for param in model.parameters():
    param.requires_grad = False

# Replace last layer for cat vs dog (2 classes)
model.fc = nn.Linear(2048, 2)

# Unfreeze last layer to train it
for param in model.fc.parameters():
    param.requires_grad = True

print("Model loaded with transfer learning!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Sahil Suryavanshi/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:42<00:00, 2.42MB/s]


Model loaded with transfer learning!
Total parameters: 23512130
Trainable parameters: 4098


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = model.to(device)

# Use CrossEntropyLoss for multi-class (better than BCELoss for this)
criterion = nn.CrossEntropyLoss()

# Only train last layer (faster)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

print("Training complete!")

Using device: cpu
Epoch [1/10], Loss: 0.6606
Epoch [2/10], Loss: 0.3783
Epoch [3/10], Loss: 0.2775
Epoch [4/10], Loss: 0.2134
Epoch [5/10], Loss: 0.1891
Epoch [6/10], Loss: 0.1655
Epoch [7/10], Loss: 0.1494
Epoch [8/10], Loss: 0.1406
Epoch [9/10], Loss: 0.1388
Epoch [10/10], Loss: 0.1518
Training complete!


In [13]:
model.eval()

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        # Get predicted class
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")

# Show sample predictions
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClasses: {train_data.classes}")

Validation Accuracy: 94.29%

Confusion Matrix:
[[22  2]
 [ 2 44]]

Classes: ['cat', 'dog']


In [14]:
torch.save(model.state_dict(),'cat_dog_model.pth')
print("Model saved!")

Model saved!


In [25]:
import cv2
import time
from ultralytics import YOLO

# Load YOLO model
model = YOLO("yolov8n.pt")

# Open webcam
cap = cv2.VideoCapture(0)

# Check camera
if not cap.isOpened():
    print("Error: Cannot access webcam")
    exit()

prev_time = 0
frame_count = 0

while True:
    ret, frame = cap.read()
    frame_count += 1
    if not ret:
        print("Failed to grab frame")
        break

    # Resize frame for better performance
    frame = cv2.resize(frame, (500, 320))

    # Run YOLO inference
    results = model(frame)
    boxes = results[0].boxes

    person_count = 0

    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        # Apply confidence threshold early
        if conf < 0.7:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = model.names[cls]

        # Count only persons (class 0)
        if cls == 0:
            person_count += 1

        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Label
        cv2.putText(frame, f"{label} {conf:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 255, 0), 2)

    # FPS calculation
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time) if prev_time != 0 else 0
    prev_time = curr_time

    # Display FPS
    cv2.putText(frame, f"FPS: {int(fps)}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Display people count
    cv2.putText(frame, f"People Count: {person_count}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Message logic
    if person_count == 0:
        message = "No Person Detected"
    elif person_count == 1:
        message = "Single Person Detected"
    else:
        message = "Multiple People Detected"

    cv2.putText(frame, message, (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Show output
    cv2.imshow("YOLO Real-Time System", frame)

    # Press ESC to exit
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 416x640 1 person, 1 dog, 336.3ms
Speed: 10.0ms preprocess, 336.3ms inference, 3.4ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 cat, 345.4ms
Speed: 19.3ms preprocess, 345.4ms inference, 3.0ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 cat, 344.1ms
Speed: 5.8ms preprocess, 344.1ms inference, 3.0ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 cat, 349.0ms
Speed: 6.4ms preprocess, 349.0ms inference, 3.3ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 cat, 345.5ms
Speed: 8.3ms preprocess, 345.5ms inference, 3.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 person, 340.3ms
Speed: 6.9ms preprocess, 340.3ms inference, 3.0ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 person, 1 dog, 357.8ms
Speed: 8.8ms preprocess, 357.8ms inference, 2.8ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 persons, 345.2ms
Speed: 7.0ms preprocess, 345.2ms inference, 3.6ms postprocess per ima